[![Roboflow Notebooks](https://media.roboflow.com/notebooks/template/bannertest2-2.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672932710194)](https://github.com/roboflow/notebooks)

# How to Segment Images with Segment Anything 3 (SAM3)

---

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roboflow-ai/notebooks/blob/main/notebooks/how-to-segment-images-with-segment-anything-3.ipynb) [![YouTube](https://badges.aleen42.com/src/youtube.svg)](https://www.youtube.com/watch?v=G1AEuFwQrWU) [![Roboflow](https://raw.githubusercontent.com/roboflow-ai/notebooks/main/assets/badges/roboflow-blogpost.svg)](https://blog.roboflow.com/sam3/) [![GitHub](https://badges.aleen42.com/src/github.svg)](https://github.com/facebookresearch/sam3)

SAM 3 (Segment Anything Model 3) extends the SAM series by moving from segmenting individual objects to understanding and segmenting all instances of a concept in images and videos. It introduces Promptable Concept Segmentation (PCS), where users specify a concept through short noun phrases like “striped cat” or by providing visual exemplars. The model detects, segments, and tracks every matching object, preserving identities across frames.


## Environment setup

### Configure your API keys

To pull Segment Anything 3 weights, you need a HuggingFace Access Token with approved access to the SAM 3 checkpoints.

- Request access to the SAM 3 checkpoints on the official Hugging Face [repo](https://github.com/facebookresearch/sam3).
- Open your HuggingFace Settings page. Click Access Tokens then New Token to generate a new token.
- In Colab, go to the left pane and click on Secrets (🔑). Store your HuggingFace Access Token under the name `HF_TOKEN`.








In [2]:
import os
from google.colab import userdata
from google.colab import drive
drive.mount('/content/drive')

!ls /content/drive/MyDrive/orbbec_data/

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
orbbec_data


### Check GPU availability

Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do that. In case of any problems navigate to `Edit` -> `Notebook settings` -> `Hardware accelerator`, set it to `T4 GPU`, and then click `Save`.

In [3]:
!nvidia-smi

Mon Mar 23 21:46:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P0             26W /   70W |     381MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import torch
import torchvision

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA is available:", torch.cuda.is_available())

PyTorch version: 2.10.0+cu128
Torchvision version: 0.25.0+cu128
CUDA is available: True


### Install SAM 3 and extra dependencies

In [5]:
!git clone https://github.com/facebookresearch/sam3.git
%cd sam3
!pip install -e ".[notebooks]"
%cd /content

fatal: destination path 'sam3' already exists and is not an empty directory.
/content/sam3
Obtaining file:///content/sam3
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sam3 (pyproject.toml) ... done
  Created wheel for sam3: filename=sam3-0.1.0-0.editable-py3-none-any.whl size=15366 sha256=9d399cf57399d61c93b6fc3036562380c8deb2bd26ffd70ffbfb5fb9636257a9
  Stored in directory: /tmp/pip-ephem-wheel-cache-pjoo9zvp/wheels/7c/90/be/80339bb9db8655024d6c9501da4e5efc6abbda4c897f5a6c43
Successfully built sam3
  Attempting uninstall: sam3
    Found existing installation: sam3 0.1.0
    Uninstalling sam3-0.1.0:
      Successfully uninstalled sam3-0.1.0


/content


In [1]:
!pip install -q supervision jupyter_bbox_widget

In [ ]:
!pip install open3d

### Download example data

Downloads example images. You can use these or replace them with your own images.

**<font color="red">⚠️ Restart session before running cells below.</font>**








## Load SAM3 Image Predictor

On Ampere GPUs (compute capability ≥ 8), we enable TensorFloat-32 (TF32) for matrix multiplications and convolutions. This allows PyTorch to use tensor cores to accelerate FP32 computations while maintaining similar numerical accuracy.

In [2]:
import torch

torch.autocast(device_type="cuda", dtype=torch.bfloat16).__enter__()

if torch.cuda.get_device_properties(0).major >= 8:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

In [4]:
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

model = build_sam3_image_model(bpe_path="/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz")
processor = Sam3Processor(model, confidence_threshold=0.3)

GatedRepoError: 403 Client Error. (Request ID: Root=1-69c1b828-71f6de4246e464a84d1d231d;7a45f493-ae95-41e2-9b42-009dfdd95044)

Cannot access gated repo for url https://huggingface.co/facebook/sam3/resolve/main/config.json.
Your request to access model facebook/sam3 is awaiting a review from the repo authors.

## Few utils to parse and visualize the result

In [ ]:
import supervision as sv

def from_sam(sam_result: dict) -> sv.Detections:
    xyxy = sam_result["boxes"].to(torch.float32).cpu().numpy()
    confidence = sam_result["scores"].to(torch.float32).cpu().numpy()

    mask = sam_result["masks"].to(torch.bool)
    mask = mask.reshape(mask.shape[0], mask.shape[2], mask.shape[3]).cpu().numpy()

    return sv.Detections(
        xyxy=xyxy,
        confidence=confidence,
        mask=mask
    )

In [ ]:
import supervision as sv
from PIL import Image
from typing import Optional


COLOR = sv.ColorPalette.from_hex([
    "#ffff00", "#ff9b00", "#ff8080", "#ff66b2", "#ff66ff", "#b266ff",
    "#9999ff", "#3399ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
])


def annotate(image: Image.Image, detections: sv.Detections, label: Optional[str] = None) -> Image.Image:
    text_scale = sv.calculate_optimal_text_scale(resolution_wh=image.size)

    mask_annotator = sv.MaskAnnotator(
        color=COLOR,
        color_lookup=sv.ColorLookup.INDEX,
        opacity=0.6
    )
    box_annotator = sv.BoxAnnotator(
        color=COLOR,
        color_lookup=sv.ColorLookup.INDEX,
        thickness=1
    )
    label_annotator = sv.LabelAnnotator(
        color=COLOR,
        color_lookup=sv.ColorLookup.INDEX,
        text_scale=0.4,
        text_padding=5,
        text_color=sv.Color.BLACK,
        text_thickness=1
    )

    annotated_image = image.copy()
    annotated_image = mask_annotator.annotate(annotated_image, detections)
    annotated_image = box_annotator.annotate(annotated_image, detections)

    if label:
        labels = [
            f"{label} {confidence:.2f}"
            for confidence in detections.confidence
        ]
        annotated_image = label_annotator.annotate(annotated_image, detections, labels)

    return annotated_image

## SAM 3 text prompt

In [ ]:
from PIL import Image
from IPython.display import display

PROMPT = "taxi"
IMAGE_PATH = '/content/traffic_jam.jpg'

image = Image.open(IMAGE_PATH).convert("RGB")
inference_state = processor.set_image(image)
inference_state = processor.set_text_prompt(state=inference_state, prompt=PROMPT)

detections = from_sam(sam_result=inference_state)
detections = detections[detections.confidence > 0.5]

print(f"There are {len(detections)} {PROMPT} objects detected in the image.\n")
annotate(image, detections, label=PROMPT)

In [ ]:
from PIL import Image
from IPython.display import display
import open3d as o3d
import numpy as np

PROMPT="green foam"


# Create an empty point cloud
pcd = o3d.geometry.PointCloud()

In [ ]:
from PIL import Image
from IPython.display import display

PROMPT = "bird"
IMAGE_PATH = '/content/birds.jpg'

image = Image.open(IMAGE_PATH).convert("RGB")
inference_state = processor.set_image(image)
inference_state = processor.set_text_prompt(state=inference_state, prompt=PROMPT)

detections = from_sam(sam_result=inference_state)
detections = detections[detections.confidence > 0.5]

print(f"There are {len(detections)} {PROMPT} objects detected in the image.\n")
annotate(image, detections)

In [ ]:
from PIL import Image
from IPython.display import display

PROMPT = "round pill"
IMAGE_PATH = '/content/pills.jpg'

image = Image.open(IMAGE_PATH).convert("RGB")
inference_state = processor.set_image(image)
inference_state = processor.set_text_prompt(state=inference_state, prompt=PROMPT)

detections = from_sam(sam_result=inference_state)
detections = detections[detections.confidence > 0.5]

print(f"There are {len(detections)} {PROMPT} objects detected in the image.\n")
annotate(image, detections)

In [ ]:
from PIL import Image
from IPython.display import display

PROMPT = "white egg"
IMAGE_PATH = '/content/eggs.jpg'

image = Image.open(IMAGE_PATH).convert("RGB")
inference_state = processor.set_image(image)
inference_state = processor.set_text_prompt(state=inference_state, prompt=PROMPT)

detections = from_sam(sam_result=inference_state)
detections = detections[detections.confidence > 0.5]

print(f"There are {len(detections)} {PROMPT} objects detected in the image.\n")
annotate(image, detections)

In [ ]:
from PIL import Image
from IPython.display import display

PROMPT = "woman"
IMAGE_PATH = '/content/solvay_conference_1927.jpg'

image = Image.open(IMAGE_PATH).convert("RGB")
inference_state = processor.set_image(image)
inference_state = processor.set_text_prompt(state=inference_state, prompt=PROMPT)

detections = from_sam(sam_result=inference_state)
detections = detections[detections.confidence > 0.5]

print(f"There are {len(detections)} {PROMPT} objects detected in the image.\n")
annotate(image, detections, label=PROMPT)

In [ ]:
from PIL import Image
from IPython.display import display

PROMPT = "plane"
IMAGE_PATH = '/content/airport.jpg'

image = Image.open(IMAGE_PATH).convert("RGB")
inference_state = processor.set_image(image)
inference_state = processor.set_text_prompt(state=inference_state, prompt=PROMPT)

detections = from_sam(sam_result=inference_state)
detections = detections[detections.confidence > 0.6]

print(f"There are {len(detections)} {PROMPT} objects detected in the image.\n")
annotate(image, detections, label=PROMPT)

In [ ]:
from PIL import Image
from IPython.display import display

PROMPT_1 = "player in white"
PROMPT_2 = "player in blue"
IMAGE_PATH = '/content/basketball_game.jpg'

image = Image.open(IMAGE_PATH).convert("RGB")
inference_state = processor.set_image(image)

inference_state = processor.set_text_prompt(state=inference_state, prompt=PROMPT_1)
detections = from_sam(sam_result=inference_state)
detections_1 = detections[detections.confidence > 0.6]

inference_state = processor.set_text_prompt(state=inference_state, prompt=PROMPT_2)
detections = from_sam(sam_result=inference_state)
detections_2 = detections[detections.confidence > 0.6]

print(f"There are {len(detections)} {PROMPT_1} objects detected in the image.\n")
print(f"There are {len(detections)} {PROMPT_2} objects detected in the image.\n")

image = annotate(image, detections_1, label=PROMPT_1)
image = annotate(image, detections_2, label=PROMPT_2)
image

## SAM 3 box prompt

In [ ]:
import base64
from io import BytesIO
from PIL import Image

OBJECTS = ['positive', 'negative']

def encode_image_pillow(image: Image.Image) -> str:
    buffer = BytesIO()
    image.save(buffer, format="JPEG")
    image_bytes = buffer.getvalue()
    encoded = base64.b64encode(image_bytes).decode("utf-8")
    return "data:image/jpeg;base64," + encoded

When prompting SAM 3 with bounding boxes, the model expects boxes in the `xcycwh` format (`x_center`, `y_center`, `width`, `height`), and the coordinates must be normalized. The code below handles the conversion to this format.

In [ ]:
import numpy as np

def get_normalized_boxes(bboxes, label, resolution_wh):
    width, height = resolution_wh
    boxes = [
        [b["x"] + b["width"] / 2, b["y"] + b["height"] / 2, b["width"], b["height"]]
        for b in bboxes
        if b["label"] == label
    ]
    if not boxes:
        return np.empty((0, 4), dtype=np.float32)
    scale = np.array([width, height, width, height], dtype=np.float32).reshape(1,-1)
    return np.array(boxes, dtype=np.float32) / scale

In [ ]:
from PIL import Image
from jupyter_bbox_widget import BBoxWidget

image_path = '/content/birds.jpg'
image = Image.open(image_path).convert("RGB")

widget = BBoxWidget(classes=OBJECTS)
widget.image = encode_image_pillow(image)
widget

In [ ]:
xyxy_positive = get_normalized_boxes(widget.bboxes, "positive", image.size)
xyxy_negatives = get_normalized_boxes(widget.bboxes, "negative", image.size)

inference_state = processor.set_image(image)
processor.reset_all_prompts(inference_state)

for box in xyxy_positive:
    inference_state = processor.add_geometric_prompt(state=inference_state, box=box, label=True)
for box in xyxy_negatives:
    inference_state = processor.add_geometric_prompt(state=inference_state, box=box, label=False)

detections = from_sam(sam_result=inference_state)
detections = detections[detections.confidence > 0.5]
annotate(image, detections)

In [ ]:
from PIL import Image
from jupyter_bbox_widget import BBoxWidget

image_path = '/content/pills.jpg'
image = Image.open(image_path).convert("RGB")

widget = BBoxWidget(classes=OBJECTS)
widget.image = encode_image_pillow(image)
widget

In [ ]:
xyxy_positive = get_normalized_boxes(widget.bboxes, "positive", image.size)
xyxy_negatives = get_normalized_boxes(widget.bboxes, "negative", image.size)

inference_state = processor.set_image(image)
processor.reset_all_prompts(inference_state)

for box in xyxy_positive:
    inference_state = processor.add_geometric_prompt(state=inference_state, box=box, label=True)
for box in xyxy_negatives:
    inference_state = processor.add_geometric_prompt(state=inference_state, box=box, label=False)

detections = from_sam(sam_result=inference_state)
detections = detections[detections.confidence > 0.5]
annotate(image, detections)

## Load SAM 3 Interactive Image Predictor

Segment Anything Model 3 (SAM 3) predicts object masks given prompts that indicate the desired object. The model first converts the image into an image embedding that allows high quality masks to be efficiently produced from a prompt.

`SAM3InteractiveImagePredictor.predict` takes the following arguments:

- `point_coords` - `[np.ndarray or None]` - a `Nx2` array of point prompts to the model. Each point is in `(X,Y)` in pixels.
- `point_labels` - `[np.ndarray or None]` - a length `N` array of labels for the
point prompts. `1` indicates a foreground point and `0` indicates a
background point.
- `box` - `[np.ndarray or None]` - a length `4` array given a box prompt to the
model, in `[x_min, y_min, x_max, y_max]` format.
- `mask_input` - `[np.ndarray]` - a low resolution mask input to the model, typically coming from a previous prediction iteration. Has form `1xHxW`, where
for SAM, `H=W=256`.
- `multimask_output` - `[bool]` - if true, the model will return three masks.
For ambiguous input prompts (such as a single click), this will often
produce better masks than a single prediction. If only a single
mask is needed, the model's predicted quality score can be used
to select the best mask. For non-ambiguous prompts, such as multiple
input prompts, `multimask_output=False` can give better results.
- `return_logits` - `[bool]` - if true, returns un-thresholded masks logits
instead of a binary mask.
- `normalize_coords` - `[bool]` - if true, the point coordinates will be normalized to the range `[0,1]` and point_coords is expected to be wrt. image dimensions.

In [ ]:
model = build_sam3_image_model(enable_inst_interactivity=True)
processor = Sam3Processor(model, confidence_threshold=0.3)

In [ ]:
def from_sam_interactive(sam_result: tuple[np.ndarray, np.ndarray]) -> sv.Detections:
    masks, scores = sam_result

    if masks.shape[0] != 1:
        masks = np.squeeze(masks)

    return sv.Detections(
        xyxy=sv.mask_to_xyxy(masks=masks),
        mask=masks.astype(bool),
        confidence=np.squeeze(scores)
    )

In [ ]:
import numpy as np

def get_xy_points(boxes, label):
    points = [
        [b["x"] + b["width"] / 2, b["y"] + b["height"] / 2]
        for b in boxes
        if b["label"] == label
    ]
    if not points:
        return np.empty((0, 2), dtype=np.float32)
    return np.array(points, dtype=np.float32)


def get_xyxy_boxes(boxes, label):
    boxes = [
        [b["x"], b["y"], b["x"] + b["width"], b["y"] + b["height"]]
        for b in boxes
        if b["label"] == label
    ]
    if not boxes:
        return np.empty((0, 4), dtype=np.float32)
    return np.array(boxes, dtype=np.float32)

## Interactive SAM 3 box prompt

In [ ]:
from PIL import Image
from jupyter_bbox_widget import BBoxWidget

image_path = '/content/dog-2.jpeg'
image = Image.open(image_path).convert("RGB")

widget = BBoxWidget(classes=OBJECTS)
widget.image = encode_image_pillow(image)
widget

In [ ]:
default_box = [
    {'x': 172, 'y': 830, 'width': 96, 'height': 179, 'label': 'positive'},
    {'x': 162, 'y': 1044, 'width': 282, 'height': 165, 'label': 'positive'},
    {'x': 353, 'y': 723, 'width': 34, 'height': 162, 'label': 'positive'},
    {'x': 468, 'y': 888, 'width': 173, 'height': 247, 'label': 'positive'}
]

boxes = widget.bboxes if widget.bboxes else default_box
boxes = get_xyxy_boxes(boxes=boxes, label='positive')

inference_state = processor.set_image(image)

masks, scores, logits = model.predict_inst(
    inference_state,
    box=boxes,
    multimask_output=False
)

detections = from_sam_interactive((masks, scores))

box_annotator = sv.BoxAnnotator(color_lookup=sv.ColorLookup.INDEX)
mask_annotator = sv.MaskAnnotator(color_lookup=sv.ColorLookup.INDEX)

source_image = box_annotator.annotate(scene=image.copy(), detections=detections)
result_image = mask_annotator.annotate(scene=image.copy(), detections=detections)

sv.plot_images_grid(
    images=[source_image, result_image],
    grid_size=(1, 2),
    titles=['source image', 'result image']
)

## Interactive SAM 3 point prompt

In [ ]:
from PIL import Image
from jupyter_bbox_widget import BBoxWidget

image_path = '/content/dog-2.jpeg'
image = Image.open(image_path).convert("RGB")

widget = BBoxWidget(classes=OBJECTS)
widget.image = encode_image_pillow(image)
widget

In [ ]:
default_box = [
    {'x': 84, 'y': 1034, 'width': 0, 'height': 0, 'label': 'positive'},
    {'x': 705, 'y': 1252, 'width': 0, 'height': 0, 'label': 'positive'},
    {'x': 704, 'y': 1015, 'width': 0, 'height': 0, 'label': 'positive'},
    {'x': 427, 'y': 1059, 'width': 0, 'height': 0, 'label': 'positive'},
    {'x': 138, 'y': 1237, 'width': 0, 'height': 0, 'label': 'positive'},
    {'x': 274, 'y': 1143, 'width': 0, 'height': 0, 'label': 'negative'},
    {'x': 463, 'y': 1117, 'width': 0, 'height': 0, 'label': 'negative'},
    {'x': 652, 'y': 1145, 'width': 0, 'height': 0, 'label': 'negative'},
    {'x': 196, 'y': 982, 'width': 0, 'height': 0, 'label': 'negative'}
]

boxes = widget.bboxes if widget.bboxes else default_box
points_positive = get_xy_points(widget.bboxes, "positive")
points_negatives = get_xy_points(widget.bboxes, "negative")

input_point = np.vstack((points_positive, points_negatives))
input_label = np.array([1] * len(points_positive) + [0] * len(points_negatives))

inference_state = processor.set_image(image)
masks, scores, logits = model.predict_inst(
    inference_state,
    point_coords=input_point,
    point_labels=input_label,
    multimask_output=True,
)

sv.plot_images_grid(
    images=masks,
    titles=[f"score: {score:.2f}" for score in scores],
    grid_size=(1, 3),
    size=(12, 12)
)

<div align="center">
  <p>
    Looking for more tutorials or have questions?
    Check out our <a href="https://github.com/roboflow/notebooks">GitHub repo</a> for more notebooks,
    or visit our <a href="https://discord.gg/GbfgXGJ8Bk">discord</a>.
  </p>
  
  <p>
    <strong>If you found this helpful, please consider giving us a ⭐
    <a href="https://github.com/roboflow/notebooks">on GitHub</a>!</strong>
  </p>

</div>